# ANALISIS MATEMATICO | AÑO: 2026

## TRABAJO FINAL

### INTEGRANTES:
* Walter Gabriel Sotelo
* Jose Ignacio	Aizpun
* Alejandro Garcia Peña
* Nicolas Ezequiel Lastra


# TP 1: LDA/QDA y optimización matemática de modelos

# Intro teórica

## Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

## Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

## LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

## Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

## Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.

# Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [115]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

## Base code

In [116]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    # Q3: para que sirve bincount?
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [117]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # Q6: por que se usa bias=True en vez del default bias=False?
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [118]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [119]:
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori)) ##
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [120]:
class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [121]:
class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [122]:
class TensorizedChol(QDA_Chol3):
    def _fit_params(self, X, y):
        # Invocamos el método de ajuste de la clase base QDA_Chol3 para obtener L_inv y las medias de cada clase
        super()._fit_params(X, y)
        # Apilamos las inversas de los factores de Cholesky (L_invs) en un tensor único de dimensiones (k, p, p)
        # donde k es la cantidad de clases y p es la cantidad de features/predictores
        self.tensor_L_invs = np.stack(self.L_invs)
        # Apilamos los vectores de medias (means) en un tensor de dimensiones (k, p, 1)
        self.tensor_means = np.stack(self.means)

    def predict(self, X):
        # Calculamos X centrado (sin la media) restando las medias de cada clase con broadcasting
        # Introducimos una dimensión extra al inicio de X para poder operar con las medias de todas las clases
        unbiased_X = X[np.newaxis, :, :] - self.tensor_means
        
        # Calculamos el término transformado Y aplicando la inversa del factor triangular
        # Y tiene forma (k, p, n) y representa la proyección de las observaciones centradas
        Y = self.tensor_L_invs @ unbiased_X
        
        # Calculamos la matriz de productos internos
        # Esto paraleliza el cálculo sobre todas las observaciones, pero genera una matriz intermedia grande de k x n x n
        # (esto es ineficiente en memoria y tiempo para n grande)
        M = Y.transpose(0, 2, 1) @ Y
        
        # Extraemos la diagonal de la matriz de productos internos para cada clase, obteniendo el término cuadrático
        # Corresponde a la norma L2 al cuadrado de cada columna
        quad_term = np.diagonal(M, axis1=1, axis2=2)
        
        # Calculamos el logaritmo del determinante del factor de Cholesky
        # Dado que L_inv es una matriz triangular, el log-determinante es la suma de los logaritmos de los elementos de su diagonal
        diags = self.tensor_L_invs.diagonal(axis1=1, axis2=2)
        log_det = np.sum(np.log(diags), axis=1)
        
        # Calculamos la probabilidad log a posteriori (más una constante) para cada combinación clase-observación
        # Sumamos la probabilidad log a priori, el log-determinante y restamos la mitad del término cuadrático
        log_posteriori = self.log_a_priori[:, np.newaxis] + log_det[:, np.newaxis] - 0.5 * quad_term
        
        # Seleccionamos la clase con la máxima probabilidad log a posteriori para cada observación
        y_hat = np.argmax(log_posteriori, axis=0)
        
        # Retornamos las predicciones adaptadas como un vector fila (1, n)
        return y_hat.reshape(1, -1)


In [123]:
class EfficientChol(QDA_Chol3):
    def _fit_params(self, X, y):
        # Invocamos el método de ajuste de la clase base QDA_Chol3 para obtener L_inv y las medias de cada clase
        super()._fit_params(X, y)
        # Apilamos las inversas de los factores de Cholesky (L_invs) en un tensor de forma (k, p, p)
        self.tensor_L_invs = np.stack(self.L_invs)
        # Apilamos las medias de cada clase en un tensor de forma (k, p, 1)
        self.tensor_means = np.stack(self.means)

    def predict(self, X):
        # Calculamos X centrado sin las medias de cada clase usando broadcasting
        unbiased_X = X[np.newaxis, :, :] - self.tensor_means
        
        # Proyectamos las observaciones centradas en el espacio transformado
        Y = self.tensor_L_invs @ unbiased_X
        
        # Calculamos de manera eficiente el término cuadrático de interés sin construir la matriz gigante k x n x n
        # Elevamos al cuadrado cada componente de Y y sumamos a lo largo del eje de las features.
        # Esto calcula directamente la norma L2 al cuadrado de cada columna, reduciendo la complejidad espacial
        quad_term = np.sum(Y ** 2, axis=1)
        
        # Calculamos el log-determinante sumando el logaritmo de los elementos de la diagonal
        diags = self.tensor_L_invs.diagonal(axis1=1, axis2=2)
        log_det = np.sum(np.log(diags), axis=1)
        
        # Calculamos la probabilidad log a posteriori para cada clase y observación
        log_posteriori = self.log_a_priori[:, np.newaxis] + log_det[:, np.newaxis] - 0.5 * quad_term
        
        # Elegimos la clase que maximiza la probabilidad log a posteriori para cada observación
        y_hat = np.argmax(log_posteriori, axis=0)
        
        # Retornamos el vector de predicciones como vector fila (1, n)
        return y_hat.reshape(1, -1)


## Datasets

Observar que se proveen **4 datasets diferentes**, el código de ejemplo usa uno solo pero eso no significa que ustedes se limiten al mismo. También pueden usar otros datasets de su elección.

In [124]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

Nota: esta clase fue creada bastante rápido y no pretende ser una plataforma súper confiable sobre la que basarse, sino más bien una herramienta simple con la que poder medir varios runs y agregar la información.

En forma rápida, `warmup` es la cantidad de runs para warmup, `mem_runs` es la cantidad de runs en las que se mide el pico de uso de RAM y `n_runs` es la cantidad de runs en las que se miden tiempos.

La razón por la que se separan es que medir memoria hace ~2.5x más lento cada run, pero al mismo tiempo se estabiliza mucho más rápido.

**Importante:** tener en cuenta que los modelos que predicen en batch (usan `predict` directamente) deberían consumir, como mínimo, $n$ veces la memoria de los que predicen por observación.

In [125]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

## Ejemplo

In [126]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [127]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [128]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [129]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [130]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [131]:
b.bench(QDA_Chol1)

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [132]:
b.summary(QDA_Chol1)

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.36660,0.202692,3.03680,0.385909,0.982407,0.018761,0.049879,0.007682,0.050402
TensorizedQDA,0.36880,0.143061,1.26970,0.443766,0.982593,0.018372,0.030843,0.011971,0.030714
QDA_Chol1,0.52005,0.251095,2.13855,0.537291,0.985741,0.018131,0.030245,0.007751,0.030790


In [133]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.36660,0.202692,3.03680,0.385909,0.982407,0.018761,0.049879,0.007682,0.050402
TensorizedQDA,0.36880,0.143061,1.26970,0.443766,0.982593,0.018372,0.030843,0.011971,0.030714
QDA_Chol1,0.52005,0.251095,2.13855,0.537291,0.985741,0.018131,0.030245,0.007751,0.030790


In [134]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.36660,3.03680,0.982407
TensorizedQDA,0.36880,1.26970,0.982593
QDA_Chol1,0.52005,2.13855,0.985741


In [135]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.36660,0.202692,3.03680,0.385909,0.982407,0.018761,0.049879,0.007682,0.050402,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.36880,0.143061,1.26970,0.443766,0.982593,0.018372,0.030843,0.011971,0.030714,0.994035,2.391746,1.021179,0.641730
QDA_Chol1,0.52005,0.251095,2.13855,0.537291,0.985741,0.018131,0.030245,0.007751,0.030790,0.704932,1.420028,1.034742,0.991019


In [136]:
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.36660,3.03680,0.982407,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.36880,1.26970,0.982593,0.994035,2.391746,1.021179,0.641730
QDA_Chol1,0.52005,2.13855,0.985741,0.704932,1.420028,1.034742,0.991019


# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores

**Sugerencia:** combinaciones adecuadas de `transpose`, `stack`, `reshape` y, ocasionalmente, `flatten` y `diagonal` suele ser más que suficiente. Se recomienda *fuertemente* explorar la dimensionalidad de cada elemento antes de implementar las clases.

## Tensorización

En esta sección nos vamos a ocupar de hacer que el modelo sea más rápido para generar predicciones, observando que incurre en un doble `for` dado que predice en forma individual un escalar para cada observación, para cada clase. Paralelizar ambos vía tensorización suena como una gran vía de mejora de tiempos.

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?
2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.


6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.

7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?
7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.
13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.
13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

In [137]:
!python --version
!pip freeze | grep /E "scipy numpy"


Python 3.13.9


"grep" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


**Comentario:** yo utilicé los siguientes parámetros para mi run de prueba. Esto NO significa que ustedes tengan que usar los mismos, tampoco el mismo dataset. Se agregó al notebook simplemente porque fue una pregunta común en cohortes anteriores.

In [138]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


# Soluciones al TP:

### Resultados del Benchmark de metodos sin cholesky:
Como era de esperar el mejor fue efficientQDA, dado que no realiza ciclos for alguno, y optimiza los calculos usando no solo los tensores (broadcasting de numpy), sino tambien el uso de la diagonal de la matriz resultante de multiplicar la traspuesta de los means de las observaciones menos los centros de las clases y la traspuesta de la inversa de la matriz de covarianza.


### Tensorized QDA:

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?
    Tensorized paraleliza sobre las clases, cicla por cada observación y calcula la distancia al centro de la clase, en todas las clases al mismo tiempo y solo queda el paso de args_max, donde se queda con el mejor (menor distancia al centro).
2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.
   - `self.tensor_inv_cov` tiene dimensiones `(k, p, p)`.
   - `self.tensor_means` tiene dimensiones `(k, p, 1)`.

    En nunpy, cuando multiplas dos vectores que no tiene el mismo numero de filas la primer matriz que de columnas la segunda, es decir tengo un MxN y un PxM, lo que sucedes es que internamente busca llevar el valor de P a N, clonando los valores de P tantas veces como necesite. en el ejercicio como P=1 y N=K cantidad de clases, clona los valores por las K clases y las multiplica todo en paralelo obteniendo los N resultados al mismo tiempo!, lo que QDA hace en un segundo ciclo for en serie Tensorized lo hace en paralelo aprovechando el producto matricial de numPY.

    1. Se calcula `unbiased_x = x - self.tensor_means`. Por broadcasting, $x$ de dimensiones `(p, 1)` se expande en la primera dimensión a `(k, p, 1)`, de manera que se restan las medias de cada clase de forma independiente.
    2. Se calcula `inner_prod = unbiased_x.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_x`. 
        Acá es donde vemos la ventaja de nunpy agregando un copia de cada Clase para ser multiplicada por la observación en una sola operación.
    3. `inner_prod.flatten()` convierte el resultado a un vector unidimensional de forma `(k,)`.
    4. El determinante se calcula en lote: `LA.det(self.tensor_inv_cov)` calcula el determinante de cada matriz `(p, p)`, resultando en un vector de forma `(k,)`. Este calculo se hace para cada X que se pasa por parametro al método
    5. El término de probabilidad condicional de QDA se calcula a la vez para todas las clases:
        `0.5 * np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()` (de forma `(k,)`).
    Aprovechando el broadcasting que realizo numpy en el punto 2
    6. En `_predict_one`, se suma `self.log_a_priori` (de forma `(k,)`) y se busca el argmax. Como las operaciones matemáticas y la lógica algebraica son idénticas a las de `QDA`, la predicción es la misma.

### 3) Faster QDA

In [139]:
## Faster QDA

class FasterQDA(TensorizedQDA):
    def predict(self, X):
        # X original tiene shape: (p, n) -> p: features, n: observaciones
        p, n = X.shape
        k = len(self.log_a_priori)
        
        # 1. Expandimos X a (1, p, n) para restar las medias (k, p, 1) usando broadcasting
        # unbiased_X resulta en shape: (k, p, n)
        unbiased_X = X.reshape(1, p, n) - self.tensor_means
        
        # 2. Multiplicación matricial de tensores: (k, p, p) @ (k, p, n)
        # A resulta en shape: (k, p, n)
        A = self.tensor_inv_cov @ unbiased_X
        
        # =====================================================================
        #  4) PUNTO CRÍTICO: Aquí aparece el monstruo de n x n
        # Transponemos unbiased_X a (k, n, p) y lo multiplicamos por A (k, p, n)
        # full_interaction resulta en un shape masivo de: (k, n, n)
        # =====================================================================
        full_interaction = unbiased_X.transpose(0, 2, 1) @ A
        
        # 4. Extraemos únicamente la diagonal principal de cada clase: shape (k, n)
        inner_prod = np.diagonal(full_interaction, axis1=1, axis2=2)
        
        # 5. Calculamos el log-condicional resolviendo el determinante: shape (k, n)
        log_det = np.log(np.linalg.det(self.tensor_inv_cov)).reshape(-1, 1)
        log_conditional = 0.5 * log_det - 0.5 * inner_prod
        
        # 6. Sumamos la probabilidad a priori: (k, 1) + (k, n) -> shape (k, n)
        log_posteriori = self.log_a_priori.reshape(-1, 1) + log_conditional
        
        # 7. Obtenemos el índice de la clase máxima para cada observación: shape (n,)
        y_hat = np.argmax(log_posteriori, axis=0)
        
        # Devolvemos como vector fila (1, n) tal cual lo pide la clase base
        return y_hat.reshape(1, -1)

### 4) Marcado como punto critico dentro del código 

### 6) Optimized QDA

In [140]:
## Optimized QDA

class EfficientQDA(TensorizedQDA):
    def predict(self, X):
        # X original tiene shape: (p, n)
        p, n = X.shape
        
        # 1. Resta de medias con broadcasting: shape (k, p, n)
        unbiased_X = X.reshape(1, p, n) - self.tensor_means
        
        # 2. Multiplicación matricial: (k, p, p) @ (k, p, n) -> shape (k, p, n)
        A = self.tensor_inv_cov @ unbiased_X
        
        # =====================================================================
        # OPTIMIZACIÓN: Aplicamos la propiedad demostrada diag(A * B)
        # Transponemos ambas matrices a shape (k, n, p)
        # Multiplicamos elemento a elemento (*) manteniendo el tamaño en (k, n, p)
        # =====================================================================
        hadamard_prod = unbiased_X.transpose(0, 2, 1) * A.transpose(0, 2, 1)
        
        # 4. Colapsamos el eje de las features (axis=2) sumando sus columnas.
        # inner_prod resulta directamente en shape (k, n) sin pasar jamás por n x n
        inner_prod = np.sum(hadamard_prod, axis=2)
        
        # 5. El resto del flujo matemático se mantiene igual que en FasterQDA
        log_det = np.log(np.linalg.det(self.tensor_inv_cov)).reshape(-1, 1)
        log_conditional = 0.5 * log_det - 0.5 * inner_prod
        
        log_posteriori = self.log_a_priori.reshape(-1, 1) + log_conditional
        y_hat = np.argmax(log_posteriori, axis=0)
        
        return y_hat.reshape(1, -1)
    

### 7) Ejecutamos el bechmark de los dos nuevos QDA elaborados y los dados en el tp

In [141]:
# bencheamos un par
to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

## Aplicación de Cholesky

8. **Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?**

    La factorizacion de Cholesky, consiste en descomponer una matriz cuadrada simétrica ($A = A^T$) y definida positiva ($x^TAx > 0$) como el     producto de una matriz triangular inferior y su matriz transpuesta $A=LL^T$. La inversa quedaria
    
    $A^{-1} = (LL^T)^{-1}$
   
    $A^{-1} = (L^T)^{-1}(L)^{-1}$

    $A^{-1} = (L^{-1})^T(L)^{-1}$


    En cuanto al metodo de Análisis Discriminante Cuadrático (QDA), la analogia con la matriz A seria la matriz de covarianzas la cual   siempre es         simetrica y definida positiva (con la premisa de que no existe multicolinealidad perfecta entre variables). Quedandonos de la siguiente       manera

    ${\Sigma}_j = L_jL_j^T$

    $\Sigma_{j}^{-1} = (L_jL_j^T)^{-1}$

   
    $\Sigma_{j}^{-1} = (L_j^T)^{-1}(L_j)^{-1}$


    $\Sigma_{j}^{-1} = (L_j^{-1})^T(L_j)^{-1}$

    Ya que el QDA utiliza la distancia de Mahalanobis, la cual contiene la forma cuadrática $(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)$

    $$
    \log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} {(x-\mu_j)^T \color{red}\Sigma_j^{-1}\color{black} (x- \mu_j)} {+ C}
    $$

    $$
    \log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} {(x-\mu_j)^T \color{red}(L_j^{-1})^T(L_j)^{-1}\color{black}(x- \mu_j)}{+ C}
    $$

    $$
    \log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} \color{red}\left[ L_j^{-1}(x- \mu_j) \right]^T L_j^{-1}(x- \mu_j)\color{black} + C
    $$

    Se puede definir un vector intermedio $y$ tal que $y=L_j^{-1}(x- \mu_j)$ que al ser una matriz triangular inferior podemos resolver de   manera directa y rapida por sustitucion hacia adelante sin calcular la inversa viendola de la siguiente forma $L_jy=(x- \mu_j)$ para luego simplificar la expresion anterior como el producto escalar de un vector consigo mismo (equivalente a su norma 2 o euclidea al cuadrado).

    $$
    \log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} \color{red}\left\|y\right\|^2\color{black}+C
    $$

    Ademas existe una optimizacion en el calculo del determinante ya que por propiedades de determinantes, el determinante de un producto es el producto de los determinantes, y al ser la transpuesta igual a la matriz original el determinante de una matriz triangular es simplemente el producto de sus elementos en la diagonal principal, quedando

    $|{\Sigma}_j| = |L_jL_j^T| = |L_j||L_j^T| = |L_j|^{2} = \left( \prod_{i=1}^{p} l_{ii} \right)^2 = \prod_{i=1}^{p} l_{ii}^2$

    aplicando logaritmo se convierte en una sumatoria quedando


   $\log|{\Sigma}_j| = 2\sum_{i=1}^{p} \log(l_{ii})$

    La expresion final para el QDA nos quedaria

   
    $$
    \log{f_j(x)} = -\color{red}\sum_{i=1}^{p} \log(l_{ii})\color{black} - \frac{1}{2} \color{red}\left\|y\right\|^2\color{black}+C
    $$


9. **Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.**

   En principio, ambas clases `QDA` y `QDA_Chol1` en su funcion de entrenamiento "_fit_params" calculan las matrices de covarianzas para cada clase,      pero QDA lo hace invirtiendo toda la matriz entera de covarianza para cada clase y guardandola, mientras que QDA_Chol1  si bien calcula las        matrices    de covarianzas, previo a la inversion factoriza la matriz de covarianzas trabajando con la triangular inferior L, lo cual es mas sencillo.

    Por otro lado en cuanto a la prediccion logaritmica condicional propiamente dicha, `QDA` como primer ineficiencia tiene el calculo del determinante de la matriz inversa de covarianzas, mientras que en `QDA_Chol1` el termino del determinante es solamente la productoria logaritmica de los componentes de la diagonal, reduciendo el costo de procesamiento del mismo. En cuanto a su segundo termino, el `QDA` presenta una infeciencia ya que lo aborda como un triple producto matricial mientras que el `QDA_Chol1` hay un paso intermedio de la multiplicacion la inversa de la matriz triangular por el unbiesd_x y su salida es simplemente un vector elevado al cuadrado.

   
    El como hace `QDA_Chol1` paso a paso para llegar hasta las predicciones es:
    
    1.  **Centrado:** Se calcula el vector de diferencia $\boldsymbol{\delta}_j = \mathbf{x} - \boldsymbol{\mu}_j$.
    2.  **Transformación:** Se calcula el vector transformado $\mathbf{y}_j = \mathbf{L}_j^{-1} \boldsymbol{\delta}_j$.
    3.  **Cálculo de la norma:** La forma cuadrática se obtiene calculando la suma de los cuadrados de los elementos de $\mathbf{y}_j$, que es $\|\mathbf{y}_j\|_2^2 = \mathbf{y}_j^T \mathbf{y}_j$. Como demostramos anteriormente, esto es matemáticamente equivalente a la forma cuadrática original.
    4.  **Cálculo del determinante:** El término log-determinante también se simplifica. El determinante de una matriz triangular es el producto de sus elementos diagonales como demostramos anteriormente en el punto 8 por lo que el calculo del determinante `LA.det(inv_cov)` se reemplaza por `L_inv.diagonal().prod()`.

    
    En resumen, `QDA_Chol1` reemplaza operaciones matriciales complejas por operaciones con matrices triangulares, que son más rápidas y estables.


10. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
Las diferencias entre las implementaciones de `QDA_Chol` dependen de cómo se utiliza la factorización de Cholesky para calcular la inversa de la matriz de covarianzas y cómo se implementa la predicción.

- `QDA_Chol1` usa la función inversa genérica `LA.inv`, la cual es muy general y versátil a la hora de calcular la inversa de una matriz. Sin embargo, al no ser un método tan específico para matrices triangulares, es más lento que otros métodos que aprovechan la estructura triangular de la matriz de Cholesky.

- `QDA_Chol2` solo calcula el factor de Cholesky, sin aplicar niguna función inicialmente y tampoco realiza inversiones de matrices en el fit. Al no calcular la inversa, ahorra tiempo en esfuerzo computacional y ejecuta la funcion `solve_triangular` que está optimizada para resolver sistemas de ecuaciones lineales con matrices triangulares, aunque esto lo hace en cada predicción.

- `QDA_Chol3` toma un enfoque similar a `QDA_Chol1` al dar `L_inv` a Cholesky, pero utiliza la función `dtrtri` la cual se especializa en calcular la inversa de una matriz triangular inferior. Por lo que es más eficiente que el método de `LA.inv` utilizado en `QDA_Chol1` al ser una función más específica.

11. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

    Lo que podemos ver es que la implementación de `EfficientQDA` es la más eficiente de momento. Es más eficientes que `QDA` y `TensorizedQDA` porque elimina el ciclo for en el método predict, lo cual quita mucha información innecesaria. Esto permite que esta implementación sea más rápida al predecir múltiples observaciones al mismo tiempo. Se apoya en el uso de la propiedad de la diagonal para evitar la matriz de `n x n` y así reducir bastante el tiempo de predicción.

    ¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?

    De las Cholesky, `QDA_Chol1` y `QDA_Chol3` tienen un rendimiento muy similar, pero `QDA_Chol3` es ligeramente más rápida en el fit ya que utiliza la función `dtrtri` para invertir la matriz triangular, por lo que su fase de preparación es más rápida que las otras implementaciones. El planteamiento de ambas es multiplicando $L_j^{-1} u$ y luego sumando los cuadrados hace que el peso computacional se reduzca considerablemente.

    ¿Alguna que sea peor?

    Podemos definir como peor a `QDA_Chol2` siendo la más lenta de las tres porque usa la función `solve_triangular` de manera poco eficiente ya que en cada una de las 4000 iteraciones del bucle for añade una sobrecarga administrativa y de llamadas inmensa que penaliza fuertemente el tiempo, dejándola como la peor de las tres.

In [142]:
# Bencheamos todas las 9 variantes de QDA implementadas
to_bench = [
    QDA,
    TensorizedQDA,
    FasterQDA,
    EfficientQDA,
    QDA_Chol1,
    QDA_Chol2,
    QDA_Chol3,
    TensorizedChol,
    EfficientChol
]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

...

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

## Resultados de Benchmarking y Análisis de Performance

Los modelos se evaluaron sobre el dataset **`letter`** de OpenML, el cual posee $n = 20000$ observaciones y $p = 16$ características (features), distribuidas en $k = 26$ clases (letras del alfabeto). La partición de test utilizada fue del $20\%$, lo que genera un tamaño de test de $n_{test} = 4000$ observaciones.

A continuación se presenta el resumen de los resultados de benchmarking obtenidos sobre esta máquina:

| Modelo | Fit Median (ms) | Predict Median (ms) | Accuracy | Peak Train Memory (MB) | Peak Test Memory (MB) | Test Speedup | Test Mem Reduction |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **QDA (Baseline)** | 4.15 | 662.67 | 0.8856 | 0.27 | 0.10 | 1.00x | 1.00x |
| **TensorizedQDA** | 4.15 | 135.00 | 0.8864 | 0.27 | 0.15 | 4.91x | 0.64x |
| **FasterQDA** | 12.03 | 540.66 | 0.8854 | 0.27 | 3199.34 | 1.23x | ~0.00x |
| **EfficientQDA** | 3.19 | 7.06 | 0.8846 | 0.27 | 38.99 | 93.86x | 0.0025x |
| **QDA_Chol1** | 4.57 | 385.14 | 0.8848 | 0.27 | 0.09 | 1.72x | 1.04x |
| **QDA_Chol2** | 3.59 | 902.85 | 0.8847 | 0.27 | 0.10 | 0.73x | 1.03x |
| **QDA_Chol3** | 3.39 | 381.07 | 0.8854 | 0.27 | 0.09 | 1.74x | 1.04x |
| **TensorizedChol** | 8.00 | 780.72 | 0.8850 | 0.27 | 3200.98 | 0.85x | ~0.00x |
| **EfficientChol** | 3.10 | 5.83 | 0.8853 | 0.27 | 38.99 | **113.61x** | 0.0025x |

### Análisis y Conclusiones de Performance

1. **El impacto de la Vectorización y la trampa de `FasterQDA`/`TensorizedChol`**:
   Los resultados muestran que la paralelización ingenua de observaciones (`FasterQDA` y `TensorizedChol`) resulta sumamente perjudicial. Al formar la matriz de interacción $n \times n$ para calcular el término cuadrático de todas las observaciones simultáneamente, se requiere computar y almacenar un tensor de dimensiones $k \times n \times n$ ($26 \times 4000 \times 4000$).
   - Este tensor requiere $26 \times 4000 \times 4000 \times 8$ bytes $\approx \mathbf{3.2\text{ GB}}$ de memoria RAM, lo cual se ve reflejado en el pico de memoria de prueba (3199 MB).
   - Debido al alto costo de transferir esta enorme cantidad de datos en memoria y la complejidad temporal de $O(k n^2 p)$ en lugar de $O(k n p)$, `FasterQDA` (540 ms) y `TensorizedChol` (780 ms) terminan siendo más lentos que sus equivalentes no vectorizados sobre observaciones o incluso que el baseline con bucles en el caso de Cholesky.

2. **La superioridad de `EfficientQDA` y `EfficientChol`**:
   Al emplear la propiedad del producto Hadamard para extraer solo la diagonal de forma directa (`np.sum(A * B_T, axis=2)`), `EfficientQDA` y `EfficientChol` eliminan por completo la matriz de interacción intermedia de tamaño $n \times n$.
   - **`EfficientQDA`** logra completar la predicción en solo **7.06 ms** (un speedup de **94x** respecto al baseline).
   - **`EfficientChol`** alcanza el mejor tiempo de todo el benchmark: **5.83 ms** (un speedup de **113x** respecto a `QDA`).
   - El pico de memoria se reduce a tan solo **38.99 MB** (unas 82 veces menos que `FasterQDA` y `TensorizedChol`), lo cual demuestra que esta optimización matemática es fundamental para hacer factible la paralelización a gran escala.

3. **Análisis de las variantes Cholesky**:
   - **`QDA_Chol1`** y **`QDA_Chol3`** (ambas en torno a los **380 ms**) logran un rendimiento superior al de `QDA` baseline (662 ms). Esto se debe a que la multiplicación $L_j^{-1} u$ y posterior suma de cuadrados es computacionalmente más liviana que la multiplicación por la matriz de covarianza completa $u^T \Sigma_j^{-1} u$, y el cálculo del determinante vía la suma del logaritmo de la diagonal es mucho más óptimo que calcular `LA.det(inv_cov)`.
   - **`QDA_Chol3`** es marginalmente más rápido en el fit debido al uso de la rutina especializada LAPACK **`dtrtri`** para invertir la matriz triangular.
   - **`QDA_Chol2`** (**902.85 ms**) es la variante no vectorizada más lenta. Aunque evita calcular la inversa de la matriz de covarianza o de su factor Cholesky, realizar un solver triangular mediante `solve_triangular` en cada una de las 4000 iteraciones del bucle de Python añade una sobrecarga administrativa y de llamadas inmensa que penaliza fuertemente el tiempo.

4. **Opinión y Relación con la Teoría**:
   Los resultados se condicen perfectamente con lo esperado teóricamente. Confirman que en lenguajes interpretados como Python/NumPy, el costo de las iteraciones en bucle es muy alto, por lo que la vectorización total es crucial. Sin embargo, vectorizar "a la fuerza" sin considerar la complejidad espacial y el uso de caché del hardware es ineficiente. El uso de la factorización de Cholesky aporta una base matemáticamente robusta y estable para la resolución de sistemas lineales y el cálculo de determinantes, y al acoplarse con la vectorización eficiente de memoria (`EfficientChol`), se obtiene el clasificador QDA óptimo en rendimiento y estabilidad numérica.

